# Week 9 Lab 2 — Shock-aligned DeepONet for a rarefied micro-nozzle

<!-- MIE690A article-aligned validation v4 -->

<!-- FLOWMLLAB_COLAB_LAUNCH_V1 -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ehsan-Roohi/FlowMLLab/blob/main/notebooks/week09/W9_Lab2_Shock_Aligned_Nozzle_DeepONet_Student.ipynb)

**Runtime:** CPU, normally under 4 minutes. **Prerequisites:** DSMC sampling,
POD/SVD, case-wise validation, and DeepONet branch--trunk notation.

This lab uses compact full-field and centerline derivatives of **all 15 real
public DSMC snapshots**
from the Roohi--Mahdavi article *Shock-centered low-rank structure and
shock-aligned surrogate modeling of rarefied micro-nozzle flows* (Physics of
Fluids 38, 082008, 2026; DOI `10.1063/5.0343101`). It asks why a moving shock
is high-rank in laboratory coordinates and low-rank in shock-centered coordinates.

### Learning outcomes

1. audit snapshot provenance and source hashes before learning;
2. detect a centerline compression station and identify noisy outliers;
3. reproduce the 15-snapshot physical versus shock-centered density POD audit;
4. select POD rank by leave-one-case-out development error;
5. open pressures 16, 25, and 30 kPa only after freezing the model;
6. generate fresh 2-D predictions for all six article outputs; and
7. compare classroom baselines and article tables as separate evidence levels.


## Evidence and claim contract

- `nozzle_fields_15cases.npz` and `nozzle_centerline_15cases.npz` are derived
  from the public Tecplot DSMC files at pinned commit
  `e1b234ba499408d3b6224633972f939f3b2301d6` and remain CC BY 4.0.
- The POD spectrum is directly reproducible from those data.
- The notebook first predicts jump-normalized centerline density, then compares
  physical-coordinate and shock-aligned interpolation baselines for density,
  $U$, $V$, temperature, Mach, and pressure. Both use the same 12 development
  fields; the shock surface for a query is interpolated only from source fields.
- The article metric tables are immutable retained evidence with a different
  output domain and must not be merged numerically with notebook errors.
- Shock centering uses target-derived locations in the structural POD and
  normalized-profile audit. For deployment, a shock-location model using only
  input pressure and training cases is separately tested below.


In [ ]:
# FLOWMLLAB_COLAB_BOOTSTRAP_V1
from pathlib import Path as _FlowMLLabPath
import os as _flowmllab_os
import subprocess as _flowmllab_subprocess
import sys as _flowmllab_sys

if "google.colab" in _flowmllab_sys.modules or _flowmllab_os.environ.get("COLAB_RELEASE_TAG"):
    _flowmllab_root = _FlowMLLabPath("/content/FlowMLLab")
    if not (_flowmllab_root / ".git").is_dir():
        _flowmllab_subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Ehsan-Roohi/FlowMLLab.git", str(_flowmllab_root)],
            check=True,
        )
    _flowmllab_subprocess.run(
        [_flowmllab_sys.executable, "-m", "pip", "install", "-q", "-e", str(_flowmllab_root)],
        check=True,
    )
    _flowmllab_os.chdir(_flowmllab_root / "notebooks/week09")

from pathlib import Path
import json
import platform
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

REPO_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "results/mahdavi_deeponet").is_dir()
)
if str(REPO_ROOT) not in _flowmllab_sys.path:
    _flowmllab_sys.path.insert(0, str(REPO_ROOT))
RESULTS = REPO_ROOT / "results/mahdavi_deeponet"
plt.rcParams.update({"font.size": 11, "axes.labelsize": 12, "legend.fontsize": 9})
print("Python:", platform.python_version())
print("FlowMLLab root:", REPO_ROOT)


In [ ]:
from sklearn.exceptions import ConvergenceWarning, DataConversionWarning
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from flowmllab.mahdavi_deeponet import (
    NOZZLE_HELD_OUT_KPA,
    density_snapshot_matrix,
    load_nozzle_centerlines,
    pod_spectrum,
    relative_l2,
    validate_week9_evidence,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=DataConversionWarning)
report = validate_week9_evidence(REPO_ROOT)
public_report = {
    "status": report["status"],
    "nozzle_cases": report["nozzle_cases"],
    "held_out_pressures_kpa": report["held_out_pressures_kpa"],
    "physical_density_pod": report["physical_density_pod"],
    "shock_centered_density_pod": report["shock_centered_density_pod"],
    "full_field_validation": {
        "status": report["nozzle_flowmllab_validation"]["status"],
        "development_pressures_kpa": report["nozzle_flowmllab_validation"][
            "development_pressures_kpa"
        ],
        "held_out_pressures_kpa": report["nozzle_flowmllab_validation"][
            "held_out_pressures_kpa"
        ],
        "selected_method_by_field": report["nozzle_flowmllab_validation"][
            "selected_method_by_field"
        ],
    },
}
print(json.dumps(public_report, indent=2))


## 1. Data audit before model fitting

The source files span back pressures from 15 to 33 kPa. Each main zone is
101 by 31; this teaching archive retains the exported max-$y$ boundary row and
seven recorded fields. `pressure_tecplot` keeps the original file values
without inferring a unit not stated by the Tecplot header.

The compact file is not a new simulation. Its provenance records every source
filename and SHA-256, the exact source commit, the extraction rule, the derived
file hash, and the licensing change notice.

**The exported symmetry values are not boundary-consistent.** The max-$y$ row
lies at $y=92\,\mu$m, the stated symmetry plane, but its raw $V$ reaches about
$-4.31$ m/s at 16 kPa. The physical boundary requires $V=0$. The archived
Fortran exporter assigns adjacent cell values to boundary nodes without odd
reflection of $V$; this is consistent with the observed defect, although the
exact producing executable for these files has not been recovered. Matching
that curve measures agreement with an exported label, not physical validity.
The original arrays remain unchanged for provenance. The separate
`predict_with_symmetry(..., symmetry_y_m=92e-6)` interface imposes the known
boundary on predictions, leaves interior values unchanged, and does not
contribute to the reported raw-data accuracy improvement. Repairing the exact
DSMC export and checking the interior remain necessary.


In [ ]:
data = load_nozzle_centerlines(REPO_ROOT)
pressure = data["pressure_kpa"]
x_um = data["x_m"] * 1e6
held_out = np.isin(pressure, NOZZLE_HELD_OUT_KPA)
development = ~held_out

audit = pd.DataFrame({
    "pressure_kPa": pressure.astype(int),
    "split": np.where(held_out, "held out", "development"),
    "shock_x_um": data["shock_x_m"] * 1e6,
    "delta_jump_um": data["delta_jump_m"] * 1e6,
})
display(audit)
print("density array:", data["density"].shape)
print("held-out pressures:", pressure[held_out].astype(int).tolist())
assert pressure[held_out].astype(int).tolist() == [16, 25, 30]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
colors = plt.cm.viridis(np.linspace(0, 1, len(pressure)))
for index, (p_value, color) in enumerate(zip(pressure, colors)):
    axes[0].plot(x_um, data["density"][index], color=color, lw=1.4, label=f"{p_value:g}")
    axes[1].scatter(p_value, data["shock_x_m"][index] * 1e6, color=color, s=36)
axes[0].set(xlabel=r"$x$ ($\mu$m)", ylabel="density (source units)", title="Real DSMC centerline snapshots")
axes[1].set(xlabel="back pressure (kPa)", ylabel=r"detected $x_s$ ($\mu$m)", title="Moving compression station")
axes[0].grid(alpha=.25)
axes[1].grid(alpha=.25)
axes[0].legend(title="kPa", ncol=3, fontsize=7, frameon=False)
plt.show()


The detector chooses the largest smoothed interior density gradient, then
estimates a jump width $\delta_j=|\rho_L-\rho_R|/|\partial\rho/\partial x|_{x_s}$.
At 22 kPa the automatic station is visibly non-monotone. Retaining that point
is important: a shock sensor is a model component that also needs validation,
not an infallible preprocessing oracle.


## 2. Structural audit: moving discontinuity versus aligned structure

For each case, define

$$
\xi_j=\frac{x-x_s}{\delta_j}, \qquad
\widetilde\rho=\frac{\rho-\rho_R}{\rho_L-\rho_R}.
$$

First use all 15 cases only for the published **representation audit**, not for
model selection. This answers how compact the snapshot family is after a known
coordinate transform. It does not estimate held-out model accuracy.


In [ ]:
pod_rows = []
matrices = {}
grids = {}
for coordinate in ("physical", "shock_centered"):
    grid, matrix = density_snapshot_matrix(
        data["x_m"], data["density"], data["shock_x_m"], data["delta_jump_m"],
        coordinate=coordinate,
    )
    spectrum = pod_spectrum(matrix)
    grids[coordinate], matrices[coordinate] = grid, matrix
    pod_rows.append({
        "coordinate": coordinate,
        "E1_percent": spectrum["first_mode_percent"],
        "E12_percent": spectrum["first_two_percent"],
        "E123_percent": spectrum["first_three_percent"],
        "N99": spectrum["n99"],
    })
pod_observed = pd.DataFrame(pod_rows)
pod_reference = pd.read_csv(RESULTS / "nozzle_pod_reference.csv")
display(pod_observed)
display(pod_reference)

fig, ax = plt.subplots(figsize=(7.4, 4.2), constrained_layout=True)
for coordinate, matrix in matrices.items():
    cumulative = pod_spectrum(matrix)["cumulative_energy"] * 100
    ax.plot(np.arange(1, len(cumulative) + 1), cumulative, "o-", label=coordinate)
ax.axhline(99, color="black", ls="--", lw=1, label="99%")
ax.set(xlabel="number of POD modes", ylabel="cumulative energy (%)", ylim=(75, 101), xlim=(1, 10))
ax.grid(alpha=.25)
ax.legend(frameon=False)
plt.show()

assert pod_observed.set_index("coordinate").loc["physical", "N99"] == 8
assert pod_observed.set_index("coordinate").loc["shock_centered", "N99"] == 2


## 3. POD--DeepONet teaching analog with a frozen split

The POD modes act as a fixed trunk, $t_k(\xi)$. A one-input neural branch maps
back pressure to modal coefficients, $b_k(P_b)$. The predicted normalized
centerline profile is

$$
\widehat{\widetilde\rho}(P_b,\xi)=\bar\rho(\xi)+
\sum_{k=1}^{r} b_k(P_b)t_k(\xi).
$$

We select rank separately in each coordinate system using leave-one-case-out
error across the 12 development pressures. Every fold recomputes the POD basis
without its validation case. Architecture, optimizer, random seed, and
candidate ranks are matched.


In [ ]:
def make_branch():
    return make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs",
            alpha=0.01, max_iter=1500, random_state=690,
        ),
    )


def fit_predict_pod_branch(matrix, train_indices, query_indices, rank):
    train_matrix = matrix[train_indices]
    mean = train_matrix.mean(axis=0)
    _, _, modes = np.linalg.svd(train_matrix - mean, full_matrices=False)
    rank = min(rank, len(train_indices) - 1)
    coefficients = (train_matrix - mean) @ modes[:rank].T
    target = coefficients.ravel() if rank == 1 else coefficients
    branch = make_branch().fit(pressure[train_indices, None], target)
    predicted_coefficients = np.asarray(
        branch.predict(pressure[query_indices, None])
    ).reshape(len(query_indices), rank)
    return mean + predicted_coefficients @ modes[:rank]


development_indices = np.flatnonzero(development)
candidate_ranks = {
    "physical": [2, 4, 6, 8],
    "shock_centered": [1, 2, 3, 4],
}
selection_rows = []
for coordinate, ranks in candidate_ranks.items():
    matrix = matrices[coordinate]
    for rank in ranks:
        fold_errors = []
        for validation_index in development_indices:
            fold_train = development_indices[development_indices != validation_index]
            prediction = fit_predict_pod_branch(
                matrix, fold_train, np.array([validation_index]), rank
            )[0]
            fold_errors.append(100 * relative_l2(matrix[validation_index], prediction))
        selection_rows.append({
            "coordinate": coordinate,
            "rank": rank,
            "LOO_mean_percent": np.mean(fold_errors),
            "LOO_max_percent": np.max(fold_errors),
        })
selection = pd.DataFrame(selection_rows)
display(selection)
selected_rank = (
    selection.sort_values("LOO_mean_percent")
    .groupby("coordinate", as_index=False).first()
    .set_index("coordinate")["rank"].astype(int).to_dict()
)
print("selected ranks:", selected_rank)


## Stop: held-out pressure gate

The representation, split, branch architecture, seed, rank candidates, and
selection statistic are now frozen. Predict which pressure will be hardest.
Then open 16, 25, and 30 kPa exactly once.


In [ ]:
held_out_indices = np.flatnonzero(held_out)
blind_rows = []
blind_predictions = {}
for coordinate in ("physical", "shock_centered"):
    prediction = fit_predict_pod_branch(
        matrices[coordinate], development_indices, held_out_indices,
        selected_rank[coordinate],
    )
    blind_predictions[coordinate] = prediction
    for local_index, case_index in enumerate(held_out_indices):
        blind_rows.append({
            "coordinate": coordinate,
            "rank": selected_rank[coordinate],
            "pressure_kPa": int(pressure[case_index]),
            "normalized_centerline_relative_L2_percent": 100 * relative_l2(
                matrices[coordinate][case_index], prediction[local_index]
            ),
        })
blind_metrics = pd.DataFrame(blind_rows)
display(blind_metrics)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.7), constrained_layout=True, sharey=True)
for axis, local_index, case_index in zip(axes, range(3), held_out_indices):
    axis.plot(grids["shock_centered"], matrices["shock_centered"][case_index],
              color="black", lw=2.2, label="DSMC target")
    axis.plot(grids["shock_centered"], blind_predictions["shock_centered"][local_index],
              color="#C44536", lw=1.8, ls="--", label="POD + neural branch")
    axis.set(title=f"held out: {pressure[case_index]:g} kPa", xlabel=r"$\xi_j$")
    axis.grid(alpha=.25)
axes[0].set_ylabel(r"jump-normalized $\rho$")
axes[0].legend(frameon=False)
plt.show()


These are one-dimensional, jump-normalized representation errors. They are
useful for comparing coordinate choices under a matched teaching model, but
they are **not commensurate** with the paper's global 2-D raw-field errors.
The expected pattern is also scientifically richer than “alignment always
wins”: average aligned error decreases, yet the 30 kPa case can remain hard.
Inspecting individual cases prevents a favorable mean from hiding that failure.


## 4. Can the shock location be predicted without target CFD?

The structural audit used target-derived $x_s$. A deployable operator cannot
read the unseen density gradient first. Fit an intentionally transparent
pressure-to-location baseline using the 12 development cases only, then test
the three frozen pressures. This is a separate component-level validation.


In [ ]:
shock_locator = make_pipeline(
    StandardScaler(), PolynomialFeatures(2), Ridge(alpha=1e-6)
).fit(pressure[development, None], data["shock_x_m"][development] * 1e6)
predicted_shock_um = shock_locator.predict(pressure[held_out, None])
shock_table = pd.DataFrame({
    "pressure_kPa": pressure[held_out].astype(int),
    "DSMC_detected_xs_um": data["shock_x_m"][held_out] * 1e6,
    "training_only_predicted_xs_um": predicted_shock_um,
})
shock_table["absolute_error_um"] = abs(
    shock_table["training_only_predicted_xs_um"] - shock_table["DSMC_detected_xs_um"]
)
display(shock_table)


The 22 kPa detector outlier remains in development data, so it can degrade this
simple locator. Do not delete it silently. Defensible next steps are to audit
the sensor window, quantify detector uncertainty, predeclare a robust fit, and
repeat the frozen test—not to tune directly on 16, 25, or 30 kPa.


## 5. Generate full-field interpolation baselines from FlowMLLab code

The following cell runs the repository's validation program. It compares plain
physical-coordinate interpolation with row-wise shock-aligned interpolation.
For each field, development-only LOO folds select alignment when it lowers
shock-window error and increases global error by no more than two percentage
points. The rule is frozen before evaluating 16, 25, and 30 kPa. These are
interpolation baselines, not a newly trained DeepONet.

The program writes a machine-readable selection table, 18 held-out rows with
global, shock-window, and density-gradient-weighted metrics for both methods,
an error heatmap, a 6-by-3 centerline comparison, and full 2-D comparisons.
Article numbers remain in their own immutable CSV; no difference column is
computed between non-commensurate evidence levels.

In the centerline figure, the $V$ panels show the prescribed symmetry condition
$V=0$, not the inconsistent exported boundary samples. Raw $V$ curves remain
in the separate symmetry audit below. No learned accuracy or relative error
against a zero boundary profile is claimed; all CSV regression metrics still
refer to the original raw fields.


In [ ]:
command = [
    _flowmllab_sys.executable,
    str(REPO_ROOT / "qa/run_nozzle_field_validation.py"),
    "--root", str(REPO_ROOT),
]
_flowmllab_subprocess.run(command, check=True)
generated_metrics = pd.read_csv(RESULTS / "nozzle_flowmllab_heldout_metrics.csv")
display(generated_metrics.pivot(
    index="field", columns="held_out_pressure_kpa",
    values="selected_global_relative_l2_percent",
))
display(generated_metrics.pivot(
    index="field", columns="held_out_pressure_kpa",
    values="selected_shock_window_relative_l2_percent",
))

try:
    from IPython.display import Image as _NozzleImage
except ModuleNotFoundError:
    _NozzleImage = None
if _NozzleImage is not None:
    for filename in (
        "nozzle_flowmllab/nozzle_back_pressure_P16_contours.png",
        "nozzle_flowmllab/nozzle_back_pressure_P25_contours.png",
        "nozzle_flowmllab/nozzle_back_pressure_P30_contours.png",
        "nozzle_flowmllab/nozzle_back_pressure_profiles.png",
        "nozzle_flowmllab/nozzle_back_pressure_error_summary.png",
    ):
        display(_NozzleImage(filename=str(RESULTS / filename)))


## 6. Registered POD models: a trained neural branch and a stronger baseline

The previous sensor sometimes selects the outlet gradient near the wall.
Translating an entire row also moves the fixed throat. The new experiment
searches for a **positive interior compression** and registers the field with
a monotone map that fixes inlet, throat, and outlet. A robust quadratic predicts
the compression station from training pressures. The 22 kPa development case
is retained; its non-monotone station is not deleted or relabelled.

The registered, channel-scaled fields define a joint POD basis. Complete-case
development folds compare ranks 2, 4, and 6, a quadratic coefficient branch,
and tanh neural branches of width 8 or 16 with three seeds. Registration,
normalization, and POD are recomputed inside each fold. Selection minimizes
the mean six-field relative $L_2$ error. The polynomial branch wins; it must
not be labelled a neural-network result. The separately trained neural ensemble
is also retained, with convergence flags and seed dispersion.

The following cell uses frozen numerical checkpoints and reproduces the
predictions without retraining. For the full development sweep, run
`python qa/run_nozzle_transport_validation.py --stage all` from the repository
root. The 16/25/30 kPa cases were inspected during earlier project work: these
new results are a **regression assessment on historical holdouts**, not a fresh
blind test. A new independent pressure sweep is needed for a research claim.


In [ ]:
from flowmllab.nozzle_transport import (
    load_transport_model, predict_transport_pod, predict_with_symmetry,
)

transport_dir = REPO_ROOT / "results/nozzle_transport"
transport_report = json.loads((transport_dir / "report.json").read_text())
transport_metrics = pd.read_csv(transport_dir / "regression_metrics.csv")
transport_model = load_transport_model(transport_dir / "selected_model.npz")
transport_prediction = predict_transport_pod(transport_model, [16.0, 25.0, 30.0])
with np.load(transport_dir / "predictions.npz", allow_pickle=False) as stored:
    np.testing.assert_allclose(transport_prediction, stored["selected"], rtol=1e-9, atol=1e-10)
print("Selected coefficient branch:", transport_report["selected_branch"])
print("Selected POD rank:", transport_report["selected_rank"])
print("Neural ensemble rank/width:", transport_report["neural_rank"], transport_report["neural_width"])
display(transport_metrics.pivot(index="field", columns="pressure_kpa",
                              values="selected_global_relative_l2_percent"))
display(transport_metrics.pivot(index="field", columns="pressure_kpa",
                              values="neural_global_relative_l2_percent"))
display(pd.read_csv(transport_dir / "physical_diagnostics.csv"))
symmetry_prediction = predict_with_symmetry(transport_model, [16, 25, 30], symmetry_y_m=92e-6)
np.testing.assert_array_equal(symmetry_prediction[:, -1, :, 2], 0.0)
print("Raw-label regression is not physical validation:", transport_report["physical_validation_passed"])
display(_NozzleImage(filename=str(transport_dir / "symmetry_boundary_audit.png")))
if _NozzleImage is not None:
    for pressure_kpa in (16, 25, 30):
        display(_NozzleImage(filename=str(transport_dir / f"nozzle_P{pressure_kpa}_profiles.png")))


The average full-field error decreases from 6.429% for the previous selected
interpolation baseline to 4.199% for registered POD with a quadratic branch;
the trained neural ensemble reaches 4.604%. At 25 kPa, the selected model's
$U$ and $V$ errors are 2.173% and 2.713%. The 30 kPa case remains difficult:
the pressure-only shock locator misses the compression station by 3.431
micrometres on average, with $U$ and Mach errors of 11.601% and 13.667%.
Neither a sharper colormap nor a target-assisted shift would resolve this
deployment limitation. Predictions are not clipped or smoothed after scoring.

The mass-flow profile error is also recorded (1.14%, 3.06%, and 2.07% for the
selected model); this does not mean exact conservation. Even the source DSMC
fields have about 6% streamwise mass-flow spread under this discrete diagnostic.
The historical shock-window definition is retained for matched comparisons;
the independently corrected station error is reported separately. The displayed
$V$ centerline uses the separately stored boundary-constrained prediction;
this plotting correction leaves model weights, raw fields, and scores unchanged.
To rebuild just these panels, run
`python qa/run_nozzle_transport_validation.py --stage profiles`.

## 7. Retained full-model paper evidence

The next tables are read from immutable CSV transcriptions. They describe the
article's full two-dimensional held-out outputs and hard 16 kPa comparison;
the notebook did not regenerate them.


In [ ]:
paper_fields = pd.read_csv(RESULTS / "nozzle_paper_field_errors.csv")
paper_baselines = pd.read_csv(RESULTS / "nozzle_hard_case_baselines.csv")
display(paper_fields.pivot(
    index="field", columns="held_out_pressure_kpa",
    values="reported_relative_l2_percent",
))
display(paper_baselines)


The retained hard-case table shows why global error alone is insufficient. In
the final article's three-seed comparison, the Cartesian Hadamard branch/trunk
model has $34.95\pm20.06\%$ shock-window error. Adding the reduced signed
distance lowers it to $9.12\pm1.01\%$, comparable to the strong Cartesian MLP's
$8.86\pm1.26\%$. The defensible claim is therefore not universal superiority.
The signed-distance representation removes a major translation burden for the
branch/trunk model; the final article explicitly does not claim a new fusion
architecture.

The method is physics-guided through representation, localized weighting, and
shock-envelope features. It does **not** thereby become a PDE-constrained
model: the reported formulation does not explicitly enforce a PDE residual,
Rankine--Hugoniot jump, or global conservation law.

### Required submission

1. the data/provenance audit and held-out pressure list;
2. physical and aligned POD energy tables;
3. leave-one-case-out rank selection;
4. per-pressure held-out profile and shock-location errors;
5. the fresh 2-D FlowMLLab contour and full-field error table;
6. one documented failure or detector ambiguity; and
7. an explicit sentence distinguishing notebook-generated evidence from the
   paper's retained full-model evidence.

### Extension

Add conservation and mass-flux diagnostics to the two baselines, then implement
a genuinely trained six-output operator using the same frozen split. A larger
network is not automatically stronger when the number of independent CFD cases
remains small.
